# GPT-2 from Scratch

This notebook is a teaching-first walkthrough of the GPT-2 architecture built in PyTorch.
We build the model one component at a time, explain every design choice, and train a tiny version on Shakespeare.

GPT-2 is the architecture that launched the modern era of large language models. Understanding it first makes every subsequent architecture (Llama, Mistral, Gemma, etc.) a clear set of diffs from this baseline.

**Relationship to the simple transformer notebook:**
If you have already worked through the simple transformer notebook in this repository, this will feel very familiar. The two architectures are nearly identical. GPT-2 is the formalized name for the decoder-only transformer design we built there. This notebook re-implements it under the canonical GPT-2 class names (`GPT2Config`, `GPT2`) and adds teaching notes that compare our choices to the original 2019 paper.

## Audience
- Learners who know basic Python.
- Students who have seen tensors and matrix multiplication before.
- Anyone who wants a readable bridge between "I know the idea of a transformer" and "I can implement GPT-2."

## Prerequisites
- Python functions, classes, and loops.
- Basic PyTorch concepts such as tensors and modules.
- A rough idea of next-token prediction.

## Learning Goals
By the end of this notebook, you should be able to:
1. Explain how GPT-2 assembles token embeddings, learned positional embeddings, multi-head attention, MLPs, and residual connections.
2. Compare our implementation to the original GPT-2 paper and name the differences (pre-norm vs post-norm, RMSNorm vs LayerNorm, ReLU-squared vs GELU, no bias, untied weights).
3. Build a data pipeline that turns raw text into `(input, target)` training pairs using byte-level tokenization.
4. Read a complete training loop with learning-rate scheduling, gradient clipping, and checkpoints.
5. Understand how autoregressive text generation works one token at a time.

## Outline

1. Set up the environment and imports.
2. Define `GPT2Config` -- the configuration object that controls model size.
3. Build the model one component at a time: RMSNorm, Multi-Head Attention, MLP, Block, full GPT2.
4. Test the model with a small smoke test.
5. Build the data pipeline and verify the tokenizer and batches.
6. Add training utilities and a full training loop.
7. Run a lightweight pipeline preview.
8. Add text generation.
9. Optional checkpoint demo.
10. Exercise, common pitfalls, and next steps.

## 1. Setup

This install cell keeps the notebook easy to run in a fresh environment such as Colab.
If your environment already has these packages, rerunning this cell is harmless.

In [ ]:
!pip install torch numpy requests -q

## 2. Imports

We gather the core tools once so every later cell can stay focused on one idea.

Teaching note:
- `dataclass` gives us a clean config object.
- `Path` helps keep file paths portable between local Jupyter and Colab-style environments.
- `torch.nn` holds reusable neural network layers.
- `torch.nn.functional` contains tensor-level math functions like `softmax` and `cross_entropy`.

In [ ]:
import math
import os
import time
from dataclasses import dataclass
from pathlib import Path

import requests
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Configuration

### `GPT2Config`

This dataclass collects the high-level knobs that control how large the model is.
Putting these values in one place makes experiments much easier because we can change width, depth, context length, or dropout without editing the model internals.

Key ideas:
- `vocab_size=256` because we tokenize raw UTF-8 bytes. The original GPT-2 used a BPE tokenizer with ~50,257 tokens.
- `n_embd=64` is the width of the hidden representation. The original GPT-2 Small used 768; we keep it tiny so the model trains on a laptop CPU.
- `n_head=4` splits attention into parallel subspaces. The original used 12 heads.
- `n_layer=4` controls depth. The original had 12 layers.
- `seq_len=256` is the maximum context window. The original GPT-2 used 1024.
- `dropout=0.0` keeps the tutorial deterministic and simple.

**How this differs from the simple transformer notebook:**
The fields are identical. We just rename the config class to `GPT2Config` to match the canonical architecture name. If you built the simple transformer first, this is the same object under a new label.

In [ ]:
@dataclass
class GPT2Config:
    vocab_size: int = 256       # byte-level tokenization
    n_embd: int = 64            # embedding dimension (small for CPU training)
    n_head: int = 4             # number of attention heads
    n_layer: int = 4            # number of transformer blocks
    seq_len: int = 256          # maximum sequence length
    dropout: float = 0.0        # dropout rate

config = GPT2Config()
print(config)

## 4. Model Building Blocks

We now build the transformer from the inside out.
The order matters for teaching:
1. normalization,
2. attention,
3. feed-forward network,
4. full block,
5. full GPT-2 model.

That mirrors how the final architecture is assembled.

**Our version vs the original GPT-2 paper (Radford et al., 2019):**

| Component | Original GPT-2 | Our version | Why we differ |
|---|---|---|---|
| Normalization | Post-norm LayerNorm | **Pre-norm RMSNorm** | Pre-norm is more stable to train; RMSNorm is simpler and works just as well when there are no biases |
| Activation | GELU | **ReLU-squared** | Sparser than GELU, smoother gradient than plain ReLU, and trivial to implement |
| Linear bias | Yes (all linear layers) | **No bias** | Modern practice shows bias is unnecessary and removing it simplifies the code |
| Weight tying | Embedding and lm_head share weights | **Untied** | Untied weights give the model more capacity and are standard in newer architectures |
| Positional encoding | Learned absolute | **Learned absolute** (same) | This is the one thing we keep identical |
| Attention | Multi-Head Attention (MHA) | **MHA** (same) | Every head has its own Q, K, V -- the standard GPT-2 design |

### `RMSNorm`

`RMSNorm` is a simpler alternative to `LayerNorm`.
Instead of subtracting the mean and dividing by the standard deviation, it only rescales the vector by its root-mean-square magnitude.

Why this is worth learning:
- it keeps activations numerically stable,
- it is slightly simpler than `LayerNorm` (no mean subtraction, no beta parameter),
- it appears in most modern transformer designs (Llama, Gemma, Mistral all use RMSNorm).

Formula:
`RMSNorm(x) = x / sqrt(mean(x^2) + eps) * gamma`

Watch for the shape logic: we normalize across the last dimension because that dimension stores the embedding features for each token.

**Compared to the original GPT-2:** the paper used `LayerNorm` in post-norm position (after the residual add). We use `RMSNorm` in pre-norm position (before the sublayer). Pre-norm is more stable during training and has become the default in all modern architectures.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight

### `MultiHeadAttention`

This class is the heart of GPT-2.
Each token creates three views of itself:
- a **query**: what information am I looking for?
- a **key**: what information do I contain?
- a **value**: what information should I pass forward?

The attention scores are computed with:
`softmax(QK^T / sqrt(d_k))`

Teaching notes:
- We split the embedding dimension across multiple heads. Each head gets `head_dim = n_embd // n_head` dimensions.
- We use a lower-triangular causal mask so a token cannot peek into the future.
- The output projection (`W_o`) mixes information from all heads back together.
- Total attention parameters: `4 * n_embd^2` (three input projections + one output projection).

Shape story to keep in mind:
`(batch, time, channels)` becomes `(batch, heads, time, head_dim)` during attention, then returns to `(batch, time, channels)` at the end.

**This is standard Multi-Head Attention (MHA)** -- every head has its own Q, K, and V projections. Compare with Grouped Query Attention (GQA) in Llama, where K and V are shared across groups of heads to reduce memory at inference time.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config: GPT2Config):
        super().__init__()
        assert config.n_embd % config.n_head == 0, "n_embd must be divisible by n_head"

        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head

        self.W_q = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.W_k = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.W_v = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.W_o = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.dropout = nn.Dropout(config.dropout)

        causal_mask = torch.tril(torch.ones(config.seq_len, config.seq_len))
        self.register_buffer(
            "causal_mask",
            causal_mask.view(1, 1, config.seq_len, config.seq_len),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        # Project to queries, keys, values
        q = self.W_q(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = attn.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        # Combine heads and project output
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.W_o(out)

### `MLP`

After attention mixes information across tokens, the MLP transforms each token representation independently.
You can think of attention as the communication step and the MLP as the per-token computation step.

Important teaching points:
- The hidden layer is expanded to `4 * n_embd`, which is a common transformer design pattern. This expansion gives the network more capacity in its nonlinear transformation.
- We use `ReLU(x)^2` (called ReLU-squared) instead of GELU. ReLU-squared is sparser than GELU (more outputs are exactly zero) and has a smoother gradient than plain ReLU near the origin.
- The projection back down to `n_embd` returns us to the residual stream width.
- No bias in any linear layer -- this is a modern simplification that works fine in practice.

**Compared to the original GPT-2:** the paper used GELU activation. GELU is slightly smoother everywhere, but ReLU-squared trains just as well for our purposes and is easier to implement from scratch.

**What changes in Llama:** the simple expand-activate-compress pattern here becomes a *gated* feed-forward network called SwiGLU, which uses two parallel up-projections with a gating mechanism.

In [ ]:
class MLP(nn.Module):
    def __init__(self, config: GPT2Config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.c_fc(x)
        x = F.relu(x).square()     # ReLU² activation
        x = self.c_proj(x)
        return self.dropout(x)

### `Block`

A transformer block combines two sublayers:
1. causal multi-head self-attention,
2. feed-forward MLP.

Both are wrapped in residual connections, and both receive normalized inputs first.
This pattern is called **pre-norm**.

Why residual connections matter:
- they give gradients a direct path through the network,
- they let each sublayer learn a correction to the current representation,
- they make deep models much easier to train.

The data flow through one block is:
```
x -> RMSNorm -> MultiHeadAttention -> + (residual) -> RMSNorm -> MLP -> + (residual) -> out
|___________________________________|                |___________________|
```

In [ ]:
class Block(nn.Module):
    def __init__(self, config: GPT2Config):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd)
        self.attn = MultiHeadAttention(config)
        self.ln_2 = RMSNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

### `GPT2`

This class assembles the full language model.
The forward pass has a clear story:
1. Turn token IDs into vectors with a token embedding table (`wte`).
2. Add learned positional embeddings (`wpe`) so order matters.
3. Pass the result through a stack of transformer blocks.
4. Normalize one last time with a final `RMSNorm`.
5. Project to vocabulary logits with `lm_head`.

Two teaching details are especially important:

**Untied weights:** We use separate matrices for the input embedding (`wte`) and the output projection (`lm_head`). The original GPT-2 tied these two matrices (they shared the same weight tensor). Tying saves parameters, but untying gives the model more capacity and is the standard in newer architectures like Llama.

**Scaled residual initialization:** We initialize the output projections (`W_o` in attention, `c_proj` in MLP) with a smaller standard deviation: `0.02 / sqrt(2 * n_layer)`. This prevents the residual stream from growing too quickly with depth at the start of training. The factor of 2 accounts for the two residual additions per block (one from attention, one from MLP).

In [ ]:
class GPT2(nn.Module):
    def __init__(self, config: GPT2Config):
        super().__init__()
        self.config = config

        # Token + position embeddings (position is learned, not rotary)
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe = nn.Embedding(config.seq_len, config.n_embd)

        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Initialize weights
        self.apply(self._init_weights)
        # Scale residual projections to prevent the residual stream from growing with depth
        for name, p in self.named_parameters():
            if name.endswith("W_o.weight") or name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.seq_len, f"Sequence length {T} exceeds max {self.config.seq_len}"

        pos = torch.arange(T, device=idx.device)
        x = self.wte(idx) + self.wpe(pos)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def count_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        print(f"Total parameters: {total:,}")
        print(f"  Token embeddings (wte):    {self.wte.weight.numel():,}")
        print(f"  Position embeddings (wpe): {self.wpe.weight.numel():,}")
        for i, block in enumerate(self.blocks):
            block_params = sum(p.numel() for p in block.parameters())
            print(f"  Block {i}: {block_params:,}")
        print(f"  Final norm:  {sum(p.numel() for p in self.ln_f.parameters()):,}")
        print(f"  LM head:     {self.lm_head.weight.numel():,}")
        return total

## 5. Model Smoke Test

Before training anything, we want a quick confidence check that the tensor shapes and gradients behave as expected.

What this test demonstrates:
- the model accepts integer token IDs,
- the logits come out with shape `(batch, time, vocab_size)`,
- the random-initialization loss is in the right ballpark,
- gradients flow through the network.

Teaching note:
For a random model with a 256-way vocabulary, the loss should start near `log(256)`, which is about `5.55`.
If the initial loss is wildly different, something is wrong with the architecture or initialization.

In [ ]:
model = GPT2(config)
model.count_parameters()

# Test forward pass shape
idx = torch.randint(0, config.vocab_size, (2, 32))
logits, _ = model(idx)
print(f"\nLogits shape: {logits.shape}")

# Test loss value at random init
targets = torch.randint(0, config.vocab_size, (2, 32))
logits, loss = model(idx, targets)
print(f"Random-init loss: {loss.item():.4f}")
print(f"Expected rough baseline: {math.log(config.vocab_size):.4f}")

# Test gradient flow
loss.backward()
grad_count = sum(1 for p in model.parameters() if p.grad is not None)
print(f"Parameters with gradients: {grad_count}")

# Verify it has learned positional embeddings (unlike Llama which uses RoPE)
assert hasattr(model, "wpe"), "GPT-2 should have learned positional embeddings"
print(f"Position embeddings: learned (seq_len={config.seq_len})")

## 6. Data Pipeline

A language model learns from one very long stream of tokens.
Our job is to turn that stream into many smaller `(input, target)` examples.

In this notebook we use the simplest tokenizer possible: raw UTF-8 bytes.
That keeps the modeling ideas front and center.

The data pipeline is identical to the one in the simple transformer notebook -- there is nothing GPT-2-specific about how we prepare training data. The same byte-level tokenizer, the same Tiny Shakespeare dataset, the same batch construction.

### Dataset Paths and Constants

Notebooks are often run from different working directories.
This cell finds the project root in a way that works both when the notebook is opened from the repository root and when it is opened from the `notebook/` folder.

We also define the Tiny Shakespeare download URL here so later functions can stay focused on behavior rather than configuration.
Checkpoints for this GPT-2 notebook are saved to a `gpt2` subdirectory to keep them separate from the simple transformer checkpoints.

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "model.py").exists() and (PROJECT_ROOT.parent / "model.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "gpt2"
SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

### `encode`

`encode` converts text into integer token IDs.
Because we are using bytes, every character becomes one or more integers in the range `0..255`.

Why start with bytes?
- no tokenizer training step,
- no vocabulary-building ceremony,
- easy to inspect and reason about,
- perfect for teaching the language modeling loop.

Trade-off:
byte-level models are simple, but they usually need longer sequences than subword tokenizers to express the same text.

In [ ]:
def encode(text: str) -> list[int]:
    return list(text.encode("utf-8"))

### `decode`

`decode` reverses the tokenizer so we can turn generated token IDs back into readable text.

Teaching note:
`errors="replace"` keeps decoding robust even if the model emits byte combinations that are not valid UTF-8 sequences.
That makes debugging generation easier because the notebook can still display something instead of crashing.

In [ ]:
def decode(tokens: list[int]) -> str:
    return bytes(tokens).decode("utf-8", errors="replace")

### `download_shakespeare`

This function downloads the Tiny Shakespeare dataset once and caches it on disk.
On later runs it simply reads the local file.

Why caching matters in notebooks:
- it keeps reruns fast,
- it avoids repeated network requests,
- it makes the workflow feel deterministic.

In [ ]:
def download_shakespeare() -> str:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    filepath = DATA_DIR / "shakespeare.txt"

    if not filepath.exists():
        print(f"Downloading tiny Shakespeare to {filepath}...")
        response = requests.get(SHAKESPEARE_URL)
        response.raise_for_status()
        filepath.write_text(response.text)
        print(f"Downloaded {len(response.text):,} characters.")

    return filepath.read_text()

### `get_batch`

`get_batch` is the bridge between a giant token stream and minibatch training.
It randomly samples starting positions, slices out windows of length `seq_len`, and creates the shifted targets for next-token prediction.

Shape intuition:
- `x` contains the visible context tokens,
- `y` contains the same sequence shifted one position to the right,
- both end up with shape `(batch_size, seq_len)`.

This one-token shift is the entire supervised learning target for autoregressive language modeling.

In [ ]:
def get_batch(data: torch.Tensor, batch_size: int, seq_len: int, device: str = "cpu"):
    max_start = len(data) - seq_len - 1
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data[i : i + seq_len] for i in starts])
    y = torch.stack([data[i + 1 : i + 1 + seq_len] for i in starts])
    return x.to(device), y.to(device)

### `prepare_data`

This helper pulls the whole data pipeline together:
1. download or load the raw text,
2. encode it into bytes,
3. turn the bytes into a tensor,
4. split that tensor into train and validation segments.

Teaching choice:
the split is a simple contiguous cut, not a shuffled split.
That keeps the logic easy to read and is good enough for a tiny educational dataset.

In [ ]:
def prepare_data(val_fraction: float = 0.1):
    text = download_shakespeare()
    tokens = encode(text)
    data = torch.tensor(tokens, dtype=torch.long)
    split_idx = int(len(data) * (1 - val_fraction))
    train_data = data[:split_idx]
    val_data = data[split_idx:]
    return train_data, val_data

## 7. Data Smoke Test

We test three things here:
1. tokenizer round-trip correctness,
2. dataset size after preparation,
3. shift-by-one behavior in the training batch.

When teaching, these checks are valuable because many training bugs begin in data handling, not in the model itself.

In [ ]:
# Tokenizer round-trip
sample_text = "Hello, World!"
sample_tokens = encode(sample_text)
round_trip = decode(sample_tokens)
print(sample_tokens)
print(round_trip)
assert round_trip == sample_text

# Dataset size
train_data, val_data = prepare_data()
print(f"Total tokens: {len(train_data) + len(val_data):,}")
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens:   {len(val_data):,}")

# Batch shift-by-one check
x, y = get_batch(train_data, batch_size=4, seq_len=32)
print(f"Batch shapes: x={x.shape}, y={y.shape}")
print(f"First input preview:  {decode(x[0].tolist())!r}")
print(f"First target preview: {decode(y[0].tolist())!r}")
assert torch.equal(x[0, 1:], y[0, :-1])

## 8. Training Utilities

The next few functions make the training loop easier to read.
Splitting them out is good software design and good teaching design: each helper introduces one concept at a time.

### `get_lr`

This function implements a two-stage learning-rate schedule:
1. **warmup**: start small so early updates are stable,
2. **cosine decay**: gradually lower the learning rate so the model can settle into a better solution.

Why not use a constant learning rate?
- too high can make training explode,
- too low can make learning painfully slow,
- a schedule gives us a smoother path from exploration to convergence.

In [ ]:
def get_lr(step: int, max_lr: float, min_lr: float, warmup_steps: int, total_steps: int) -> float:
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps

    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return min_lr + (max_lr - min_lr) * cosine

### `estimate_loss`

Training loss on a single minibatch is noisy.
`estimate_loss` gives a more trustworthy picture by averaging over several fresh batches from both the training split and the validation split.

Important habit:
switch the model to `eval()` mode while measuring and back to `train()` mode afterward.
That keeps evaluation behavior consistent when models use layers such as dropout.

In [ ]:
@torch.no_grad()
def estimate_loss(model, train_data, val_data, batch_size, seq_len, device, eval_iters=20):
    model.eval()
    results = {}
    for name, data in [("train", train_data), ("val", val_data)]:
        losses = []
        for _ in range(eval_iters):
            x, y = get_batch(data, batch_size, seq_len, device)
            _, loss = model(x, y)
            losses.append(loss.item())
        results[name] = sum(losses) / len(losses)
    model.train()
    return results

### `save_checkpoint`

A checkpoint captures the current training state so you can pause work, resume later, or keep the best-performing model.

We save four pieces of information:
- the model weights,
- the optimizer state,
- the config,
- the current step.

That is enough to continue training or load the model for generation later.

In [ ]:
def save_checkpoint(model, config, optimizer, step, path):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
            "step": step,
        },
        path,
    )

### `load_checkpoint`

This is the inverse of `save_checkpoint`.
It rebuilds the model from the saved config, restores the learned weights, and returns the training step.

Teaching idea:
separating save/load logic from the training loop is a good pattern because generation code can reuse it later without copying model setup logic.

Note that we instantiate a `GPT2` model here (not the simple transformer's `GPT` class). If you have both notebooks in the same repository, the checkpoint files are kept in separate subdirectories to avoid confusion.

In [ ]:
def load_checkpoint(path, device="cpu"):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    config = checkpoint["config"]
    model = GPT2(config).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    return model, config, checkpoint["step"]

### `get_device`

This helper picks the best available accelerator.
It tries CUDA first, then Apple's MPS backend, and falls back to CPU.

In a teaching notebook, making device choice explicit helps learners understand why the same code can run on different hardware with only small changes.

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

### `sanity_check`

Before a long training run, it is wise to overfit a single fixed batch.
This is one of the most useful debugging tricks in deep learning.

What success looks like:
the loss should fall very close to zero because the model only needs to memorize one batch.

If this fails, the problem is usually fundamental:
- a bug in the model,
- a bug in the data pipeline,
- a bug in the optimization loop.

In [ ]:
def sanity_check(config, device):
    print("=" * 60)
    print("SANITY CHECK: overfitting one batch")
    print("=" * 60)

    model = GPT2(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    train_data, _ = prepare_data()
    x, y = get_batch(train_data, batch_size=4, seq_len=config.seq_len, device=device)

    for step in range(500):
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 50 == 0:
            print(f"step {step:4d} | loss {loss.item():.4f}")

    final_loss = loss.item()
    print(f"Final loss: {final_loss:.4f}")
    return final_loss < 0.1

### `train`

This is the full training loop.
It combines every previous idea:
- data loading,
- model creation,
- optimizer setup,
- learning-rate scheduling,
- gradient computation,
- gradient clipping,
- validation checks,
- checkpoint saving.

As you read it, notice the rhythm of training:
1. choose a learning rate for the current step,
2. sample a batch,
3. run forward and backward passes,
4. update the weights,
5. occasionally evaluate and save.

This cell is longer than the others because a training loop naturally integrates many concerns, but the helper functions above keep it manageable.

In [ ]:
def train(
    config,
    device,
    max_steps=2000,
    batch_size=32,
    max_lr=3e-3,
    min_lr=3e-4,
    warmup_steps=100,
    eval_interval=100,
    save_interval=500,
    checkpoint_dir=CHECKPOINT_DIR,
):
    print("=" * 60)
    print("TRAINING")
    print("=" * 60)
    print(f"Device: {device}")
    print(f"Config: n_embd={config.n_embd}, n_head={config.n_head}, n_layer={config.n_layer}")
    print(f"Steps: {max_steps}, Batch size: {batch_size}, Seq len: {config.seq_len}")
    print(f"LR: {max_lr} -> {min_lr} (warmup {warmup_steps} steps)")
    print()

    train_data, val_data = prepare_data()
    model = GPT2(config).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {total_params:,}")

    decay_params = [p for p in model.parameters() if p.dim() >= 2]
    nodecay_params = [p for p in model.parameters() if p.dim() < 2]
    optimizer = torch.optim.AdamW(
        [
            {"params": decay_params, "weight_decay": 0.1},
            {"params": nodecay_params, "weight_decay": 0.0},
        ],
        lr=max_lr,
        betas=(0.9, 0.95),
    )

    use_amp = device in ("cuda", "mps")
    scaler = torch.amp.GradScaler(enabled=(device == "cuda"))
    amp_dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float16

    os.makedirs(checkpoint_dir, exist_ok=True)
    best_val_loss = float("inf")
    t0 = time.time()

    for step in range(max_steps):
        lr = get_lr(step, max_lr, min_lr, warmup_steps, max_steps)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        x, y = get_batch(train_data, batch_size, config.seq_len, device)

        if use_amp:
            with torch.autocast(device_type=device, dtype=amp_dtype):
                _, loss = model(x, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            _, loss = model(x, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        optimizer.zero_grad(set_to_none=True)

        if step % 10 == 0:
            elapsed = time.time() - t0
            tokens_per_sec = (step + 1) * batch_size * config.seq_len / elapsed if elapsed > 0 else 0
            print(f"step {step:5d} | loss {loss.item():.4f} | lr {lr:.2e} | {tokens_per_sec:.0f} tok/s")

        if step > 0 and step % eval_interval == 0:
            losses = estimate_loss(model, train_data, val_data, batch_size, config.seq_len, device)
            print(f"eval | train {losses['train']:.4f} | val {losses['val']:.4f}")
            if losses["val"] < best_val_loss:
                best_val_loss = losses["val"]
                save_checkpoint(model, config, optimizer, step, Path(checkpoint_dir) / "best.pt")
                print("saved new best checkpoint")

        if step > 0 and step % save_interval == 0:
            save_checkpoint(model, config, optimizer, step, Path(checkpoint_dir) / "latest.pt")

    save_checkpoint(model, config, optimizer, max_steps, Path(checkpoint_dir) / "final.pt")
    print(f"Training complete. Best val loss: {best_val_loss:.4f}")
    return model

## 9. Lightweight Pipeline Preview

Instead of launching a full training run right away, this cell does a quick end-to-end pass:
- choose a device,
- build the config and model,
- load data,
- sample one batch,
- compute one forward pass.

This is a great habit in real projects because it catches integration problems early while staying fast enough for iteration.

In [ ]:
device = get_device()
config = GPT2Config()
model = GPT2(config).to(device)
train_data, val_data = prepare_data()
x, y = get_batch(train_data, batch_size=2, seq_len=32, device=device)
logits, loss = model(x, y)

print(f"Selected device: {device}")
print(f"Input batch shape:  {x.shape}")
print(f"Target batch shape: {y.shape}")
print(f"Logits shape:       {logits.shape}")
print(f"One-batch loss:     {loss.item():.4f}")

## 10. Text Generation

Training teaches the model a probability distribution over the next token.
Generation turns that distribution into actual text by repeating the same loop:
1. feed the current context into the model,
2. read the logits for the last position,
3. choose the next token,
4. append it to the context,
5. repeat.

Temperature and top-k are two simple ways to control the sampling behavior.

### `generate`

This function implements autoregressive sampling.

What each argument teaches:
- `prompt_tokens`: generation always starts from some context, even if that context is just a single dummy token.
- `max_new_tokens`: generation is iterative, so we control how many steps to take.
- `temperature`: lower values make the model more conservative; higher values make it more random.
- `top_k`: restricts sampling to the most plausible candidates and reduces wild mistakes.

A subtle but important detail:
we only feed the last `seq_len` tokens back into the model, because GPT-2 was trained with a fixed maximum context window defined by its learned positional embeddings. Unlike RoPE-based models (Llama), GPT-2 cannot extrapolate beyond this window at all.

In [ ]:
@torch.no_grad()
def generate(model, prompt_tokens: list[int], max_new_tokens: int = 200, temperature: float = 0.8, top_k: int = 50, device: str = "cpu") -> list[int]:
    model.eval()
    seq_len = model.config.seq_len
    tokens = torch.tensor([prompt_tokens], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx = tokens[:, -seq_len:]
        logits, _ = model(idx)
        logits = logits[:, -1, :]

        if temperature == 0:
            next_token = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k > 0:
                top_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < top_values[:, -1:]] = float("-inf")
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

        tokens = torch.cat([tokens, next_token], dim=1)

    return tokens[0].tolist()

## 11. Optional Checkpoint Demo

This cell tries to load a saved checkpoint and generate text.
If no checkpoint exists yet, it prints a friendly message instead of failing.

That makes the notebook safe to run top-to-bottom in a fresh clone while still showing how inference works once you have trained the model.

In [ ]:
checkpoint_path = CHECKPOINT_DIR / "best.pt"

if checkpoint_path.exists():
    demo_model, demo_config, step = load_checkpoint(checkpoint_path, device)
    prompt_tokens = encode("ROMEO:")
    output_tokens = generate(
        demo_model,
        prompt_tokens,
        max_new_tokens=200,
        temperature=0.8,
        top_k=50,
        device=device,
    )
    print(f"Loaded checkpoint from step {step}")
    print(decode(output_tokens))
else:
    print(f"No checkpoint found at {checkpoint_path}.")
    print("Run a training cell first if you want to try generation from learned weights.")

## 12. Exercise

Try this before reading the scaffold cell:

1. Create a larger `GPT2Config` with a wider embedding size and more layers.
2. Build the model.
3. Compare the parameter count to the default model.
4. Predict which change will increase memory use the most: width, depth, or sequence length.

Reflection question:
Why do you think parameter count grows much faster when width increases than when depth increases a little? (Hint: attention and MLP both have weight matrices whose size depends on `n_embd^2`.)

In [ ]:
exercise_config = GPT2Config(
    n_embd=128,
    n_head=8,
    n_layer=6,
    seq_len=256,
)

exercise_model = GPT2(exercise_config)
print("Default model:")
GPT2(GPT2Config()).count_parameters()
print("\nExercise model:")
exercise_model.count_parameters()

## 13. Common Pitfalls and Next Steps

### Common Pitfalls
- **Running cells out of order**: later function cells depend on earlier imports and class definitions.
- **Using a sequence length longer than the configured context window**: the model will assert if `T > seq_len`. With learned positional embeddings, there is no way to extrapolate beyond the trained window (unlike RoPE, which can at least attempt it).
- **Confusing token count with character count**: byte tokenization is simple, but some characters map to multiple bytes (e.g., accented characters, emoji).
- **Judging learning from one noisy batch**: use `estimate_loss` instead of trusting a single minibatch.
- **Expecting fast training on CPU**: even a tiny transformer is much more pleasant on GPU.

### Extensions
- Add dropout and compare training curves.
- Swap byte tokenization for a subword tokenizer (e.g., `tiktoken` with the GPT-2 BPE vocabulary).
- Tie the embedding and output weights (`model.lm_head.weight = model.wte.weight`) and compare parameter count.
- Replace `ReLU^2` with GELU to match the original GPT-2 paper exactly.
- Log attention maps for a short prompt and inspect which tokens attend to which earlier positions.

### What Llama changes from GPT-2
If you continue to the Llama notebook or file in this repository, here are the three main architectural diffs:
1. **Learned positional embeddings -> RoPE (Rotary Positional Embeddings)**: instead of a learned embedding table, RoPE encodes position by rotating the query and key vectors. This allows some length generalization beyond the training window.
2. **Multi-Head Attention -> Grouped Query Attention (GQA)**: instead of giving every head its own K and V projections, GQA shares K and V across groups of heads. This dramatically reduces the KV-cache memory needed during inference.
3. **ReLU-squared MLP -> SwiGLU**: the simple expand-activate-compress MLP becomes a gated network with two parallel up-projections and a `SiLU(gate) * value` mechanism, which tends to train better at scale.